## Supported and unsupported options

This notebook checks the DSCIM options currently supported by the CLI.
For options that are not supported, it shows why and points to the
relevant source.

See [demo.ipynb](demo.ipynb) for a full example from start to finish.

All examples use small generated inputs. Install the `run` extra first:

`uv pip install ".[run]"`

## Inputs

Generated inputs for both run modes, and one config per mode. The ssp
config sweeps three recipe and discounting pairs and includes a reduce
block; the rff config points at precomputed damage-function
coefficients and keeps the uncollapsed outputs for the `scc` step.

In [1]:
import pathlib
import sys

import yaml

repo = pathlib.Path.cwd().resolve()
if not (repo / "tests").exists():
    repo = repo.parent
sys.path.insert(0, str(repo / "tests"))
import fixture_factory

data = repo / "examples" / "coverage_data"
data.mkdir(exist_ok=True)

ssp = fixture_factory.ssp_fixture_config(data)
fixture_factory.write_batch_damages(data)
ssp["sweep"]["menu_pairs"] = [
    {"recipe": "adding_up", "discounting": "euler_ramsey"},
    {"recipe": "risk_aversion", "discounting": "constant"},
    {"recipe": "adding_up", "discounting": "gwr_gwr"},
]
ssp["reduce"] = {"reductions": ["cc", "no_cc"], "recipes": ["adding_up", "risk_aversion"]}
ssp_path = data / "ssp.yml"
ssp_path.write_text(yaml.safe_dump(ssp))

rff = fixture_factory.rff_fixture_config(data)
rff["scc"] = {
    "deflator": 1.0,
    "collapse": "mean",
    "output": str(data / "scghgs"),
}
rff_path = data / "rff.yml"
rff_path.write_text(yaml.safe_dump(rff))
print(f"wrote {ssp_path} and {rff_path}")

/project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/.venv/lib/python3.13/site-packages/zarr/api/asynchronous.py:246: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


wrote /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/coverage_data/ssp.yml and /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/coverage_data/rff.yml


/project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/.venv/lib/python3.13/site-packages/zarr/api/asynchronous.py:246: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


## The four SCC variants

Standard and GWR run; Quantile Regression and Regional do not. The
refusals below come from the catalogue, so the reason and the dscim
source citation appear in the output.

### Standard

risk_aversion with constant discounting. Constant discounting adds a
`discrate` dimension, one entry per rate in dscim's fixed list.

In [2]:
!dscim-cil run {ssp_path} --recipe risk_aversion --discounting constant

settings                                     
option                value           origin 
discounting_type      constant        flag   
discrete_discounting  False           default
eta                   2.0             config 
ext_method            global_c_ratio  default
fair_aggregation      ['ce', 'mean']  config 
fit_type              ols             default
gases                 ['CO2_Fossil']  config 
pulse_year            2020            config 
recipe                risk_aversion   flag   
rho                   0.0001          config 
sector                labor           config 
weitzman_parameter    [0.1]           config 


INFO running labor 2020 risk_aversion/constant eta=2.0 rho=0.0001

 Executing 
        Running risk_aversion
        sector: labor
        discounting: constant
        eta: 2.0
        rho: 0.0001
        
INFO 
 Executing 
        Running risk_aversion
        sector: labor
        discounting: constant
        eta: 2.0
        rho: 0.0001
        
Processing damage functions ...
INFO Processing damage functions ...
Existing damage functions not found. Damage points will be loaded.
INFO Existing damage functions not found. Damage points will be loaded.
Risk-aversion CEs found at /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/coverage_data/reduced/labor/risk_aversion_no_cc_eta2.0.zarr. These are being loaded...
INFO Risk-aversion CEs found at /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/coverage_data/reduced/labor/risk_aversion_no_cc_eta2.0.zarr. These are being loaded...
Risk-aversion CEs found at /project/cil/home_dirs/scadavidsanchez/repos/dscim-

Extrapolating global consumption.
INFO Extrapolating global consumption.
End-of-century growth rates are not capped.


/project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/.venv/lib/python3.13/site-packages/pyarrow/compute.py:230: FutureWarning: Specifying null_placement in RankOptions is deprecated as of 25.0.0. Specify null_placement per sort_key instead.
  return options_class(*args, **kwargs)
/project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/.venv/lib/python3.13/site-packages/pyarrow/compute.py:230: FutureWarning: Specifying null_placement in RankOptions is deprecated as of 25.0.0. Specify null_placement per sort_key instead.
  return options_class(*args, **kwargs)


/project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/.venv/lib/python3.13/site-packages/pyarrow/compute.py:230: FutureWarning: Specifying null_placement in RankOptions is deprecated as of 25.0.0. Specify null_placement per sort_key instead.
  return options_class(*args, **kwargs)
/project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/.venv/lib/python3.13/site-packages/pyarrow/compute.py:230: FutureWarning: Specifying null_placement in RankOptions is deprecated as of 25.0.0. Specify null_placement per sort_key instead.
  return options_class(*args, **kwargs)


Processing SCC calculation ...
INFO Processing SCC calculation ...


Saving /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/coverage_data/results/labor/2020/unmasked/risk_aversion_constant_eta2.0_rho0.0001_scc.nc4
INFO Saving /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/coverage_data/results/labor/2020/unmasked/risk_aversion_constant_eta2.0_rho0.0001_scc.nc4


Saving /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/coverage_data/results/labor/2020/unmasked/risk_aversion_constant_eta2.0_rho0.0001_uncollapsed_sccs.nc4
INFO Saving /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/coverage_data/results/labor/2020/unmasked/risk_aversion_constant_eta2.0_rho0.0001_uncollapsed_sccs.nc4


Results available: /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/coverage_data/results/labor/2020/unmasked
INFO Results available: /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/coverage_data/results/labor/2020/unmasked
completed: labor 2020 risk_aversion/constant eta=2.0 rho=0.0001 (metadata: /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/coverage_data/results/labor/2020/unmasked/risk_aversion_constant_eta2.0_rho0.0001_run_metadata.yaml)


In [3]:
import xarray as xr

standard = xr.open_dataset(
    data / "results" / "labor" / "2020" / "unmasked"
    / "risk_aversion_constant_eta2.0_rho0.0001_scc.nc4"
)
print(standard.scc.dims)
print(standard.discrate.values)

('discrate', 'fair_aggregation', 'weitzman_parameter', 'discount_type', 'ssp', 'model', 'rcp', 'gas')
[0.01  0.015 0.02  0.025 0.03  0.05 ]


### GWR

gwr_gwr pools the damage-function fit across ssp and model; the
collapsed coordinates become stringified lists in the output.

In [4]:
!dscim-cil run {ssp_path} --recipe adding_up --discounting gwr_gwr

settings                                     
option                value           origin 
discounting_type      gwr_gwr         flag   
discrete_discounting  False           default
eta                   2.0             config 
ext_method            global_c_ratio  default
fair_aggregation      ['ce', 'mean']  config 
fit_type              ols             default
gases                 ['CO2_Fossil']  config 
pulse_year            2020            config 
recipe                adding_up       flag   
rho                   0.0001          config 
sector                labor           config 
weitzman_parameter    [0.1]           config 


INFO running labor 2020 adding_up/gwr_gwr eta=2.0 rho=0.0001

 Executing 
        Running adding_up
        sector: labor
        discounting: gwr_gwr
        eta: 2.0
        rho: 0.0001
        
INFO 
 Executing 
        Running adding_up
        sector: labor
        discounting: gwr_gwr
        eta: 2.0
        rho: 0.0001
        
Processing damage functions ...
INFO Processing damage functions ...
Existing damage functions not found. Damage points will be loaded.
INFO Existing damage functions not found. Damage points will be loaded.
Adding up aggregated damages found at /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/coverage_data/reduced/labor/adding_up_cc.zarr, /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/coverage_data/reduced/labor/adding_up_no_cc.zarr. These are being loaded...
INFO Adding up aggregated damages found at /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/coverage_data/reduced/labor/adding_up_cc.zarr, /project/ci

Extrapolating global consumption.
INFO Extrapolating global consumption.
End-of-century growth rates are not capped.


Processing SCC calculation ...
INFO Processing SCC calculation ...
End-of-century growth rates are not capped.


End-of-century growth rates are not capped.
Saving /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/coverage_data/results/labor/2020/unmasked/adding_up_gwr_gwr_eta2.0_rho0.0001_scc.nc4
INFO Saving /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/coverage_data/results/labor/2020/unmasked/adding_up_gwr_gwr_eta2.0_rho0.0001_scc.nc4


End-of-century growth rates are not capped.


Saving /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/coverage_data/results/labor/2020/unmasked/adding_up_gwr_gwr_eta2.0_rho0.0001_uncollapsed_sccs.nc4
INFO Saving /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/coverage_data/results/labor/2020/unmasked/adding_up_gwr_gwr_eta2.0_rho0.0001_uncollapsed_sccs.nc4


Results available: /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/coverage_data/results/labor/2020/unmasked
INFO Results available: /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/coverage_data/results/labor/2020/unmasked
completed: labor 2020 adding_up/gwr_gwr eta=2.0 rho=0.0001 (metadata: /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/coverage_data/results/labor/2020/unmasked/adding_up_gwr_gwr_eta2.0_rho0.0001_run_metadata.yaml)


In [5]:
gwr = xr.open_dataset(
    data / "results" / "labor" / "2020" / "unmasked"
    / "adding_up_gwr_gwr_eta2.0_rho0.0001_scc.nc4"
)
print("ssp coordinate:", gwr.ssp.values)
print("model coordinate:", gwr.model.values)

ssp coordinate: ["[np.str_('SSP2'), np.str_('SSP3')]"]
model coordinate: ["[np.str_('m1'), np.str_('m2')]"]


### Quantile Regression

Not executable. Expected to fail:

In [6]:
!dscim-cil validate {ssp_path} -c menu.fit_type=quantreg

error: menu.fit_type 'quantreg': dscim asserts it is incompatible with risk_aversion reduction; it emits a different artifact set (full_uncertainty_iqr/stat_uncertainty_iqr instead of the standard SCC outputs) and requires batch-keeping reduced damages, a storage layout incompatible with the reduced files the run stage otherwise consumes. Production used it only for the labor and agriculture papers, never for CAMEL. [preprocessing.py:85-92 (risk_aversion assert); main_recipe.py order_plate quantreg branch] Pass --allow-unsupported to proceed anyway.


In [7]:
!dscim-cil explain fit_type quantreg

fit_type: Damage-function estimation type.
  status: supported
  stages: fit
  modes: ssp, rff
  default: 'ols'
  source: main_recipe.py:87; dispatch utils/utils.py (modeler)
  values:
    'quantreg': unsupported [specification]
        dscim asserts it is incompatible with risk_aversion reduction; it emits a different artifact set (full_uncertainty_iqr/stat_uncertainty_iqr instead of the standard SCC outputs) and requires batch-keeping reduced damages, a storage layout incompatible with the reduced files the run stage otherwise consumes. Production used it only for the labor and agriculture papers, never for CAMEL.
        source: preprocessing.py:85-92 (risk_aversion assert); main_recipe.py order_plate quantreg branch


### Regional

The regional surface exists only on dscim's generalize_df_fit branch,
not on the main branch dscim-cil targets. Expected to fail:

In [8]:
!dscim-cil validate {ssp_path} -c menu.geography=ir

error: 'geography' is not on dscim main: the regional surface exists only on the generalize_df_fit branch
error: unknown menu key 'geography'


In [9]:
!dscim-cil explain geography

geography: Regional aggregation level (ir/country/globe).
  status: removed
  reason: Absent from dscim main's MainRecipe; the regional surface exists only on the generalize_df_fit and harmonize branches.
  restriction class: specification
  stages: fit
  modes: ssp, rff
  default: None
  source: absent from main (MainRecipe.__init__ main_recipe.py:77-110); exists on generalize_df_fit and the harmonize branch


## Aggregations

The reduce stage collapses the batch dimension: mean for adding_up,
certainty equivalent for risk_aversion. Both naming conventions land in
the reduced-damages library (adding_up unsuffixed, risk_aversion
eta-suffixed, per dscim main).

In [10]:
!dscim-cil reduce {ssp_path}

/project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/.venv/lib/python3.13/site-packages/zarr/api/asynchronous.py:246: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


/project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/.venv/lib/python3.13/site-packages/zarr/api/asynchronous.py:246: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


/project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/.venv/lib/python3.13/site-packages/zarr/api/asynchronous.py:246: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


/project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/.venv/lib/python3.13/site-packages/zarr/api/asynchronous.py:246: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


completed: /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/coverage_data/reduced/labor/adding_up_cc.zarr
completed: /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/coverage_data/reduced/labor/risk_aversion_cc_eta2.0.zarr
completed: /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/coverage_data/reduced/labor/adding_up_no_cc.zarr
completed: /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/coverage_data/reduced/labor/risk_aversion_no_cc_eta2.0.zarr


In [11]:
library = pathlib.Path(ssp["paths"]["reduced_damages_library"]) / "labor"
for entry in sorted(library.iterdir()):
    print(entry.name)

adding_up_cc.zarr
adding_up_no_cc.zarr
risk_aversion_cc_eta2.0.zarr
risk_aversion_no_cc_eta2.0.zarr


The batch-keeping variant of reduction exists only for quantile
regression:

In [12]:
!dscim-cil explain quantreg

quantreg: Keep the batch dimension through reduction for quantile regression.
  status: unsupported
  reason: Only affects the adding_up branch (risk_aversion asserts against it) and keeps batch in the reduced output, a layout consumed only by quantile-regression fits (see fit_type: quantreg).
  restriction class: specification
  stages: reduce
  modes: ssp
  default: False
  source: preprocessing.py:83; adding_up-only behavior in ce_from_chunk; risk_aversion assert preprocessing.py:91-92


## Damage function fit

The fit accepts exactly dscim's twelve formulas, matched as whole
strings. A near-miss gets the nearest valid formula back. Expected to
fail:

In [13]:
!dscim-cil validate {ssp_path} -c 'sectors.labor.formula=damages ~ -1 anomaly + np.power(anomaly, 2)'

error: sector 'labor' formula 'damages ~ -1 anomaly + np.power(anomaly, 2)' is not one of dscim's 12 formulas (main_recipe.py:62-75; exact string match including whitespace) (did you mean 'damages ~ -1 + anomaly + np.power(anomaly, 2)'?)


## Extrapolation

Only global_c_ratio is implemented in dscim; time_trends belongs to an
older dscim and is catalogued as removed. Expected to fail:

In [14]:
!dscim-cil validate {ssp_path} -c menu.ext_method=time_trends

error: menu.ext_method 'time_trends': only 'global_c_ratio' is implemented in dscim; other values fall through to UnboundLocalError (utils/utils.py model_outputs)


In [15]:
!dscim-cil explain ext_method

ext_method: Extrapolation method for the damage function beyond the fit window.
  status: supported
  stages: fit
  modes: ssp, rff
  default: 'global_c_ratio'
  source: main_recipe.py:89; implementation utils/utils.py (model_outputs)
  values:
    'global_c_ratio': supported
    'time_trends': removed [library]
        Appears in one archived production config from an older dscim era; current dscim implements only global_c_ratio and any other value falls through to UnboundLocalError.
        source: utils/utils.py model_outputs (global_c_ratio only); dscim-research configs/archive/hybrid_mortality_config.yaml:15


## FaIR aggregation

ce, mean, gwr_mean, median, and median_params are valid members and run
(the variant runs above use the config's set). The literal
`uncollapsed` is not a member; that pipeline is reached with an empty
list plus the `scc` command, shown under Run modes.

In [16]:
!dscim-cil explain fair_aggregation uncollapsed

fair_aggregation: How to collapse FaIR uncertainty into SCCs. An empty list skips the collapsed-SCC computation and leaves the uncollapsed trio.
  status: supported
  stages: fair, discount
  modes: ssp, rff
  default: ('ce', 'mean', 'gwr_mean', 'median', 'median_params')
  source: main_recipe.py:101,118-119; dispatch main_recipe.py (marginal_damages)
  values:
    'uncollapsed': unsupported [library]
        Not a valid member: marginal_damages has no branch for it and raises NotImplementedError, which breaks calculate_scc. The uncollapsed pipeline is reached by fair_aggregation: [] plus the dscim-cil scc command, which composes SCCs from the uncollapsed outputs.
        source: main_recipe.py marginal_damages (no uncollapsed branch) and order_plate


## Discounting

Seven of dscim's eight discount types run; constant_gwr is listed in
dscim but its discount-factor path is unimplemented.

In [17]:
!dscim-cil explain discounting_type constant_gwr

discounting_type: Discounting scheme; also controls damage-function fit grouping and population collapse.
  status: supported
  stages: fit, discount
  modes: ssp, rff
  required in config; dscim would default to None if unset, but dscim-cil never applies that silently
  source: main_recipe.py:88 (default None); accepted set main_recipe.py:52-61; assert main_recipe.py:245-247
  values:
    'constant_gwr': unsupported [library]
        Listed in DISCOUNT_TYPES but its non-constant discount-factor path has no branch in calculate_stream_discount_factors and raises UnboundLocalError; never used in any production repo or in dscim's own test matrix.
        source: main_recipe.py:52-61; calculate_stream_discount_factors has no constant_gwr branch; dscim's tests/conftest.py discount_types fixture omits it


## Run modes

Everything above is the discrete SSP/RCP mode. The EPA/RFF mode carries
a runid dimension, consumes precomputed coefficients, skips fitting,
and keeps every draw; `scc` then composes and collapses them.

In [18]:
!dscim-cil run {rff_path}

settings                                     
option                value           origin 
discounting_type      euler_ramsey    config 
discrete_discounting  False           default
eta                   2.0             config 
ext_method            global_c_ratio  default
fair_aggregation      []              config 
fit_type              ols             default
gases                 ['CO2_Fossil']  config 
pulse_year            2020            config 
recipe                risk_aversion   config 
rho                   0.0001          config 
sector                CAMEL_test      config 
weitzman_parameter    [0.1]           config 


INFO running CAMEL_test 2020 risk_aversion/euler_ramsey eta=2.0 rho=0.0001

 Executing 
        Running risk_aversion
        sector: CAMEL_test
        discounting: euler_ramsey
        eta: 2.0
        rho: 0.0001
        
INFO 
 Executing 
        Running risk_aversion
        sector: CAMEL_test
        discounting: euler_ramsey
        eta: 2.0
        rho: 0.0001
        
Processing damage functions ...
INFO Processing damage functions ...


Processing SCC calculation ...
INFO Processing SCC calculation ...


Saving /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/coverage_data/rff_results/CAMEL_test/2020/unmasked/risk_aversion_euler_ramsey_eta2.0_rho0.0001_uncollapsed_sccs.nc4
INFO Saving /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/coverage_data/rff_results/CAMEL_test/2020/unmasked/risk_aversion_euler_ramsey_eta2.0_rho0.0001_uncollapsed_sccs.nc4


Saving /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/coverage_data/rff_results/CAMEL_test/2020/unmasked/risk_aversion_euler_ramsey_eta2.0_rho0.0001_uncollapsed_marginal_damages.nc4
INFO Saving /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/coverage_data/rff_results/CAMEL_test/2020/unmasked/risk_aversion_euler_ramsey_eta2.0_rho0.0001_uncollapsed_marginal_damages.nc4


Saving /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/coverage_data/rff_results/CAMEL_test/2020/unmasked/risk_aversion_euler_ramsey_eta2.0_rho0.0001_uncollapsed_discount_factors.nc4
INFO Saving /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/coverage_data/rff_results/CAMEL_test/2020/unmasked/risk_aversion_euler_ramsey_eta2.0_rho0.0001_uncollapsed_discount_factors.nc4


Results available: /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/coverage_data/rff_results/CAMEL_test/2020/unmasked
INFO Results available: /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/coverage_data/rff_results/CAMEL_test/2020/unmasked


completed: CAMEL_test 2020 risk_aversion/euler_ramsey eta=2.0 rho=0.0001 (metadata: /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/coverage_data/rff_results/CAMEL_test/2020/unmasked/risk_aversion_euler_ramsey_eta2.0_rho0.0001_run_metadata.yaml)


In [19]:
!dscim-cil scc {rff_path}

completed: /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/coverage_data/scghgs/CAMEL_test/2020/unmasked/risk_aversion_euler_ramsey_eta2.0_rho0.0001_scghg.nc4


In [20]:
scghg = xr.open_dataset(
    data / "scghgs" / "CAMEL_test" / "2020" / "unmasked"
    / "risk_aversion_euler_ramsey_eta2.0_rho0.0001_scghg.nc4"
)
print(dict(scghg.scghg.sizes), "->", float(scghg.scghg.squeeze()))

{'weitzman_parameter': 1, 'discount_type': 1, 'gas': 1} -> 20981.01477695062


## Step dependencies

`plan` derives the pipeline order from the config. With an empty
reduced-damages library, the run step is blocked and each missing input
names the command that produces it:

In [21]:
!dscim-cil plan {ssp_path} -c paths.reduced_damages_library={data}/empty_library

root: /project/cil/home_dirs/scadavidsanch…pos/dscim-cil/examples/coverage_data
1. reduce: collapse batch for labor  [ready]
     in  [ok] labor_damages.zarr
     in  [ok] econ.zarr
     out [new] empty_library/labor/adding_up_cc.zarr
     out [new] empty_library/labor/ri…aversion_cc_eta2.0.zarr
     out [new] empty_library/labor/adding_up_no_cc.zarr
     out [new] empty_library/labor/ri…rsion_no_cc_eta2.0.zarr
2. run: fit and integrate labor  [blocked-by-4]
     in  [ok] fair.nc
     in  [ok] conversion.nc
     in  [ok] econ.zarr
     in  [ok] gmst.csv
     in  [missing] empty_library/labor/adding_up_cc.zarr  <- dscim-cil reduce
     in  [missing] empty_library/labor/adding_up_no_cc.zarr  <- dscim-cil reduce
     in  [missing] empty_library/labor/ri…aversion_cc_eta2.0.zarr  <- dscim-cil…
     in  [missing] empty_library/labor/ri…rsion_no_cc_eta2.0.zarr  <- dscim-cil…
     out [new] results/labor/2020/unm…ta2.0_rho0.0001_scc.nc4
     out [new] results/labor/2020/unm…01_uncollapsed_sccs

Against the real library the same step is ready:

In [22]:
!dscim-cil plan {ssp_path}

root: /project/cil/home_dirs/scadavidsanch…pos/dscim-cil/examples/coverage_data
1. reduce: collapse batch for labor  [outputs-present]
     in  [ok] labor_damages.zarr
     in  [ok] econ.zarr
     out [exists] reduced/labor/adding_up_cc.zarr
     out [exists] reduced/labor/risk_aversion_cc_eta2.0.zarr
     out [exists] reduced/labor/adding_up_no_cc.zarr
     out [exists] reduced/labor/risk_aversion_no_cc_eta2.0.zarr
2. run: fit and integrate labor  [ready]
     in  [ok] fair.nc
     in  [ok] conversion.nc
     in  [ok] econ.zarr
     in  [ok] gmst.csv
     in  [ok] reduced/labor/adding_up_cc.zarr  <- dscim-cil reduce
     in  [ok] reduced/labor/adding_up_no_cc.zarr  <- dscim-cil reduce
     in  [ok] reduced/labor/risk_aversion_cc_eta2.0.zarr  <- dscim-cil reduce
     in  [ok] reduced/labor/risk_aversion_no_cc_eta2.0.zarr  <- dscim-cil reduce
     out [new] results/labor/2020/unm…ta2.0_rho0.0001_scc.nc4
     out [new] results/labor/2020/unm…01_uncollapsed_sccs.nc4
     out [exists] resu

## ECS masks

Masks were catalogued as supported until the test matrix ran one: every
masked run crashes inside dscim on current xarray. The catalogue now
carries that fact. Expected to fail:

In [23]:
!dscim-cil validate {ssp_path} -c 'sweep.masks=[keep_first]'

error: sweep mask 'keep_first': Climate.anomalies assigns the result of Dataset.update, which returns None on current xarray, so every masked run crashes with TypeError before computing anything; masking worked only with the older xarray where update returned the dataset. [simple_storage.py:56; application in Climate.anomalies; production names from dscim-research main/run_integration_result.py:183-190] Pass --allow-unsupported to proceed anyway.


In [24]:
!dscim-cil explain ecs_mask_name

ecs_mask_name: Mask variable to apply; both path and name must be set for masking to occur. Open set; five names are used in production.
  status: unsupported
  reason: Climate.anomalies assigns the result of Dataset.update, which returns None on current xarray, so every masked run crashes with TypeError before computing anything; masking worked only with the older xarray where update returned the dataset.
  restriction class: library
  stages: fair
  modes: ssp
  default: None
  source: simple_storage.py:56; application in Climate.anomalies; production names from dscim-research main/run_integration_result.py:183-190
  values (open set; known values):
    'truncate_at_ecs995symmetric_passing_mask': supported
    'truncate_at_ecs990symmetric_passing_mask': supported
    'truncate_at_ecs950symmetric_passing_mask': supported
    'truncate_at_ecs830symmetric_passing_mask': supported
    'truncate_at_ecs750symmetric_passing_mask': supported
